# Векторный поиск

Устанавливаем библиотеки через пакетный менеджер.


-U - обновление пакетов до последних версий

sentence-transformers - библиотека для работы с эмбеддингами, нужна для семантического поиска

transformers - библиотека Hugging Face для работы с трансформер-моделями

In [1]:
pip install -U sentence-transformers transformers pandas numpy scikit-learn

In [3]:
# Импортируем необходимые библиотеки
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer # библиотека для генерации эмбеддингов
from sklearn.metrics.pairwise import cosine_similarity # библиотека для вычисления и сравнения косинусного расстояния

In [4]:
DENSE_RANKINGS_PATH = "dense_rankings.csv"


## **ДЕМО**

Демо-версия для понимания процесса формирования эмбеддингов и выбора наиболее подходящего чанка.

Для этого необходимо перевести каждый чанк в эмбеддинг, перевести вопрос в эмбеддинг, вычислить косинусное расстояние и отсортировать чанки по близости к вопросу.

In [5]:
toy_chunks = [
    "Выручка компании за 2024 год составила 703 741 млн руб.",
    "Чистая прибыль выросла на 15 процентов.",
    "Капитальные затраты снизились по сравнению с прошлым годом.",
    "Количество клиентов мобильной связи увеличилось."
] # Создаем список демо-чанков

toy_chunk_ids = ["d1", "d2", "d3", "d4"] # Создаем список демо-ID чанков

toy_question = "Какая выручка компании за 2024 год?" # Демо-вопрос

In [6]:
#Для более быстрой загрузки модели воспользуемся токеном из кабинета HuggingFace
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [7]:
# Загружаем модель
model = SentenceTransformer("intfloat/multilingual-e5-large")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Для загруженной модели необходимо использовать преффиксы **query** и **passage**

**query** - для вопроса

**passage** - для чанка

In [8]:
toy_passages = [f"passage: {text}" for text in toy_chunks] # Присваиваем преффикс passage каждому чанку
toy_query = [f"query: {toy_question}"] # Присваиваем преффикс query вопросу

print(toy_query)
print(toy_passages[0])

['query: Какая выручка компании за 2024 год?']
passage: Выручка компании за 2024 год составила 703 741 млн руб.


Преобразуем тексты в числовые векторы

Метод **encode()** преобразует текст в числовые векторы (эмбеддинги)

In [9]:
# Кодируем чанки
toy_passage_embeddings = model.encode(
    toy_passages, # список чанков
    batch_size=32, # определяет, сколько текстов обрабатывается одновременно, ускоряет работу
    normalize_embeddings=True, # Приводим вектор к единичной длине
    show_progress_bar=False, # Не показывать прогресс-бар
)

# Кодируем вопросы
toy_query_embedding = model.encode(
    toy_query, # список вопросов
    batch_size=1, # Обрабатываем 1 текст за раз
    normalize_embeddings=True, # Приводим вектор к единичной длине
    show_progress_bar=False, # Не показывать прогресс-бар
)

In [10]:
# эмбеддинг вопроса на 1024 элемента (стандарт для модели)
toy_query_embedding

array([[-0.00502959,  0.001259  , -0.01846605, ...,  0.00483143,
        -0.04279127,  0.00040075]], shape=(1, 1024), dtype=float32)

In [11]:
# эмбеддинги 4 чанков на 1024 элемента (стандарт для модели)
toy_passage_embeddings

array([[ 0.01627781, -0.00793281, -0.01652344, ...,  0.01013484,
        -0.04524498,  0.00318337],
       [ 0.0291121 , -0.01273165, -0.0110264 , ..., -0.00083146,
        -0.02153984, -0.00045761],
       [ 0.02496765, -0.00453519, -0.02472969, ...,  0.00524633,
        -0.04414998, -0.01316174],
       [ 0.01591744, -0.02578917, -0.02609371, ..., -0.00090834,
        -0.01726528,  0.00681287]], shape=(4, 1024), dtype=float32)

In [12]:
# Вычисляем косинусное сходство между вектором вопроса и всеми векторами чанков
toy_scores = cosine_similarity(toy_query_embedding, toy_passage_embeddings)[0]

# Создаем таблицу с результатами
toy_results = pd.DataFrame({
    "chunk_id": toy_chunk_ids,
    "text": toy_chunks,
    "score": toy_scores,
}).sort_values("score", ascending=False).reset_index(drop=True) # сортируем по релевантности чанков
# сбрасываем старые индексы строк и создаём новые
toy_results

,chunk_id,text,score
0,d1,Выручка компании за 2024 год составила 703 741...,0.908400
1,d3,Капитальные затраты снизились по сравнению с п...,0.736221
2,d2,Чистая прибыль выросла на 15 процентов.,0.732288
3,d4,Количество клиентов мобильной связи увеличилось.,0.709866


Получаем результат и понимаем, что хорошо работает. Теперь расширяем код на реальные данные

## **Работа с реальными данными**

In [14]:
chunks_df = pd.read_csv("chunks_final.csv") #Читаем таблицу с чанками
gold_df = pd.read_csv("gold_final.csv") #Читаем таблицу gold_final

print("chunks_df:", chunks_df.shape) # Размеры таблицы с чанками
print("gold_df:", gold_df.shape) # Размеры таблицы gold_final

display(chunks_df.head())
display(gold_df.head())

chunks_df: (20630, 8)
gold_df: (49, 6)


,chunk_id,company_raw,company_slug,source_file,pdf_page,chunk_idx_on_page,text,n_chars
0,ашинский металлургический завод-p001-c001,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,1,1,ГРУППА «АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД» КОНСО...,170
1,ашинский металлургический завод-p001-c002,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,1,2,"ЗА ГОД, ЗАКОНЧИВШИЙСЯ 31 ДЕКАБРЯ 2024 г., ВО И...",177
2,ашинский металлургический завод-p002-c001,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,1,Группа «Ашинский металлургический завод» Содер...,315
3,ашинский металлургический завод-p002-c002,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,2,закончившийся 31 декабря 2024 года ..............,599
4,ашинский металлургический завод-p002-c003,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,3,1. Общие сведения о Группе и ее деятельности ....,363


,company,question,answer,pdf_page,question_id,chunk_id
0,мтс,1. Какая выручка компании за 2024 год?,703 741 млн. руб.,12,0,мтс-p012-c001
1,хэдхантер,2. Какую чистую прибыль компания получила за 2...,23 881 807 тыс. руб,8,1,хэдхантер-p009-c002
2,пао югк,3. Какова общая сумма активов компании на 31 д...,154 631 млн. руб.,10,2,пао югк-p010-c002
3,пао афк система,4. Какова сумма денежных средств и их эквивале...,157 879 млн. руб,9,3,пао афк система-p009-c003
4,транснефть,5. Какова общая сумма долгосрочных кредитов и ...,191 590 млн. руб.,5,4,транснефть-p005-c005


Проверяем качество входных данных

In [15]:
# Задаем список необходимых колонок в таблицах
required_chunk_cols = ["chunk_id", "text"]
required_gold_cols = ["question_id", "question", "chunk_id"]

# Выявляем недостающие колонки (списки будут пустыми, если все колонки на месте)
missing_chunk_cols = [c for c in required_chunk_cols if c not in chunks_df.columns]
missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]

# При отсутствии останавливаем код с ошибкой и сообщением, каких именно колонок не хватает
assert not missing_chunk_cols, f"В chunks_final.csv не хватает колонок: {missing_chunk_cols}"
assert not missing_gold_cols, f"В gold_final.csv не хватает колонок: {missing_gold_cols}"

# Выполняем предобработку данных. Преобразуем в строку, удаляем лишние пробелы и заменяем пропущенные значения на пустую строку
chunks_df["chunk_id"] = chunks_df["chunk_id"].astype(str).str.strip()
chunks_df["text"] = chunks_df["text"].fillna("").astype(str).str.strip()

# Выполняем предобработку данных. Преобразуем в строку, удаляем лишние пробелы и заменяем пропущенные значения на пустую строку
gold_df["question_id"] = gold_df["question_id"].astype(str).str.strip()
gold_df["question"] = gold_df["question"].fillna("").astype(str).str.strip()
gold_df["chunk_id"] = gold_df["chunk_id"].astype(str).str.strip()

# Удаляем пустые чанки, сбрасываем старую нумерацию строк и создаем новую
chunks_df = chunks_df[chunks_df["text"] != ""].reset_index(drop=True)

print("Проверка входных данных пройдена")

Проверка входных данных пройдена


Переходим к работе с эмбеддингами

In [16]:
# Создаем список чанков и вопросов с соответствующими префиксами
passage_texts = [f"passage: {text}" for text in chunks_df["text"].tolist()]
query_texts = [f"query: {text}" for text in gold_df["question"].tolist()]

# Для проверки:
print(passage_texts[0][:200]) # Выводим первые 200 символов первого чанка
print(query_texts[0]) # Выводим ппервый вопрос

passage: ГРУППА «АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД» КОНСОЛИДИРОВАННАЯ ФИНАНСОВАЯ ОТЧЕТНОСТЬ, ПОДГОТОВЛЕННАЯ В СООТВЕТСТВИИ С МЕЖДУНАРОДНЫМИ СТАНДАРТАМИ ФИНАНСОВОЙ ОТЧЕТНОСТИ (МСФО),
query: 1. Какая выручка компании за 2024 год?


In [17]:
# Создаем функцию для перевода текста в эмбеддинг
def encode_texts(model, texts, batch_size=32):
    """
    Кодирует список строк в embeddings.

    Аргументы:
    - model: загруженная SentenceTransformer-модель
    - texts: список строк
    - batch_size: размер батча при кодировании

    Возвращает:
    - numpy array формы (n_texts, embedding_dim)
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,  # обязательно для E5 + cosine similarity
        show_progress_bar=True # Нужно для отслеживания прогресса
    )
    return embeddings

Проверяем функцию на ДЕМО

In [18]:
toy_query_emb = encode_texts(model, toy_query) # Преобразуем вопрос в эмбеддинг
toy_pass_emb = encode_texts(model, toy_passages) # Преобразуем чанки в эмбеддинги

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
toy_query_emb.shape # Проверка размера

(1, 1024)

In [20]:
toy_pass_emb.shape # Проверка размера

(4, 1024)

Совпали значения с тем, что мы делали до этого. Можем применить функцию на наши реальные данные


Создаем эмбеддинги чанков

In [21]:
chunk_embeddings = encode_texts(model, passage_texts, batch_size=32)
print(chunk_embeddings.shape)

Batches:   0%|          | 0/645 [00:00<?, ?it/s]

(20630, 1024)


**Создаем таблицу с чанками, добавляя к ним эмбеддинги**


Создаём новую колонку "embedding", в которую преобразует каждый вектор эмбеддинга в JSON-строку, тк CSV-файлы на прямую не хранят списки/массивы.


vec.tolist() — превращает numpy-массив в обычный Python-список


json.dumps() — преобразует список в JSON-строку

ensure_ascii=False — сохраняет кириллицу как есть

In [22]:
chunks_with_emb_df = chunks_df.copy()
chunks_with_emb_df["embedding"] = [
    json.dumps(vec.tolist(), ensure_ascii=False) for vec in chunk_embeddings
]

chunks_with_emb_df.head()

,chunk_id,company_raw,company_slug,source_file,pdf_page,chunk_idx_on_page,text,n_chars,embedding
0,ашинский металлургический завод-p001-c001,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,1,1,ГРУППА «АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД» КОНСО...,170,"[0.044525034725666046, -0.018163220956921577, ..."
1,ашинский металлургический завод-p001-c002,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,1,2,"ЗА ГОД, ЗАКОНЧИВШИЙСЯ 31 ДЕКАБРЯ 2024 г., ВО И...",177,"[0.025012889876961708, -0.0025465376675128937,..."
2,ашинский металлургический завод-p002-c001,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,1,Группа «Ашинский металлургический завод» Содер...,315,"[0.03303804621100426, -0.0016106144757941365, ..."
3,ашинский металлургический завод-p002-c002,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,2,закончившийся 31 декабря 2024 года ..............,599,"[0.03177128732204437, -0.00382411852478981, -0..."
4,ашинский металлургический завод-p002-c003,АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ ЗАВОД,ашинский металлургический завод,/content/data/Дата/АШИНСКИЙ МЕТАЛЛУРГИЧЕСКИЙ З...,2,3,1. Общие сведения о Группе и ее деятельности ....,363,"[0.033201877027750015, -0.019542764872312546, ..."


In [23]:
# Сохраняем табличку чанков с эмбеддингами в CSV-табличку без индекса, тк есть ID
chunks_with_emb_df.to_csv("chunks_with_e5_embeddings.csv", index=False, encoding="utf-8-sig")

Создаем эмбеддинги вопросов из golden set.

В реальной эксплуатации запрос пользователя векторизуется в реальном времени

In [24]:
query_embeddings = encode_texts(model, query_texts, batch_size=32)
print(query_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(49, 1024)


Создаем функцию для возвращения топ-k чанков для одного вопроса

Функция retrieve_top_k_dense

    """
    Строит top-k выдачу для одного вопроса.

    Аргументы:
    - question_id: id вопроса
    - query_embedding: embedding одного вопроса
    - chunk_embeddings: embeddings всех чанков
    - chunks_df: DataFrame с чанками, где есть колонка chunk_id
    - top_k: сколько лучших чанков вернуть

    Возвращает:
    DataFrame с колонками:
    - question_id
    - method
    - rank
    - chunk_id
    - score
    """

In [30]:
def retrieve_top_k_dense(question_id, query_embedding, chunk_embeddings, chunks_df, top_k=10):
    # Считаем косинусную близость между query_embedding и всеми chunk_embeddings
    # cosine_similarity ожидает 2D массивы, поэтому оборачиваем query_embedding в список
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]

    # Сортируем чанки по убыванию score (косинусной близости)
    # argsort возвращает индексы, которые отсортировали бы массив, порядок по убыванию
    sorted_indices = np.argsort(similarities)[::-1]

    # Отбираем top_k лучших
    top_indices = sorted_indices[:top_k]

    # Собираем результат в DataFrame
    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            'question_id': question_id,
            'method': 'dense',  # помечаем метод как "dense" (семантический поиск)
            'rank': rank,
            'chunk_id': chunks_df.iloc[idx]['chunk_id'],
            'score': float(similarities[idx])  # преобразуем numpy float в Python float
        })

    return pd.DataFrame(results)

Тестируем функцию на реальных данных для первого вопроса, получаем 10 наиболее релевантных чанков с максимальным score

In [31]:
test_question_id = gold_df.iloc[0]["question_id"]
test_query_embedding = query_embeddings[0]

test_results = retrieve_top_k_dense(
    question_id=test_question_id,
    query_embedding=test_query_embedding,
    chunk_embeddings=chunk_embeddings,
    chunks_df=chunks_df,
    top_k=10,
)

display(test_results)

,question_id,method,rank,chunk_id,score
0,0,dense,1,пао омз-p042-c002,0.871737
1,0,dense,2,пао омз-p041-c005,0.870817
2,0,dense,3,глобалтрак менеджмент-p057-c007,0.870680
3,0,dense,4,мечел-p001-c002,0.869142
4,0,dense,5,лента-p064-c002,0.868623
5,0,dense,6,пао афк система-p031-c006,0.868352
6,0,dense,7,пао ятэк-p008-c002,0.867237
7,0,dense,8,мтс-p036-c004,0.867223
8,0,dense,9,русгидро-p060-c004,0.867145
9,0,dense,10,мтс-p043-c004,0.867126


Создаем функцию build_dense_rankings, которая строит итоговый ranking DataFrame для всех вопросов
   
   
    """
    Аргументы:
    - gold_df: таблица вопросов, где есть question_id
    - query_embeddings: embeddings всех вопросов в том же порядке, что и строки gold_df
    - chunk_embeddings: embeddings всех чанков
    - chunks_df: таблица чанков
    - top_k: сколько чанков возвращать на каждый вопрос

    Возвращает:
    единый DataFrame со всеми результатами retrieval
    """

In [42]:
def build_dense_rankings(gold_df, query_embeddings, chunk_embeddings, chunks_df, top_k=10):

    results_list = [] # Создаем пустой список для сбора результатов поиска для каждого вопроса

    for i in range(len(gold_df)): # Начало цикла по всем вопросам golden сета
        row = gold_df.iloc[i] # Извлекаем i-ю строку из таблицы вопросов

        # Вызываем функцию нахождения TOP-K чанков для одного вопроса
        result = retrieve_top_k_dense(
            question_id=row['question_id'],
            query_embedding=query_embeddings[i],  # i точно соответствует строке row
            chunk_embeddings=chunk_embeddings,
            chunks_df=chunks_df,
            top_k=top_k
        )

        results_list.append(result)

    # Склеиваем все результаты в один DataFrame, игнорируя индекс
    return pd.concat(results_list, ignore_index=True)

Запускаем функцию для получения итогового Датафрейма

In [43]:
#Запускаем функцию
dense_rankings_df = build_dense_rankings(
    gold_df=gold_df,
    query_embeddings=query_embeddings,
    chunk_embeddings=chunk_embeddings,
    chunks_df=chunks_df,
    top_k=10,
)

# Проверяем результат - размерность и вывод
print("dense_rankings_df:", dense_rankings_df.shape)
dense_rankings_df.head(20)

dense_rankings_df: (490, 5)


,question_id,method,rank,chunk_id,score
0,0,dense,1,пао омз-p042-c002,0.871737
1,0,dense,2,пао омз-p041-c005,0.870817
2,0,dense,3,глобалтрак менеджмент-p057-c007,0.870680
3,0,dense,4,мечел-p001-c002,0.869142
4,0,dense,5,лента-p064-c002,0.868623
5,0,dense,6,пао афк система-p031-c006,0.868352
6,0,dense,7,пао ятэк-p008-c002,0.867237
7,0,dense,8,мтс-p036-c004,0.867223
8,0,dense,9,русгидро-p060-c004,0.867145
9,0,dense,10,мтс-p043-c004,0.867126


Проверяем полученные данные

In [44]:
# Задаем список необходимых колонок в таблице
required_output_cols = ["question_id", "method", "rank", "chunk_id", "score"]

# Выявляем недостающие колонки (списки будут пустыми, если все колонки на месте)
missing_output_cols = [c for c in required_output_cols if c not in dense_rankings_df.columns]

# При отсутствии останавливаем код с ошибкой и сообщением, каких именно колонок не хватает
assert not missing_output_cols, f"В dense_rankings_df не хватает колонок: {missing_output_cols}"

# Останавливаем код при обнаружении следующих ошибок:
assert (dense_rankings_df["method"] == "dense").all(), "Колонка method должна быть равна 'dense'"
assert (dense_rankings_df["rank"] >= 1).all(), "rank должен начинаться с 1"
assert dense_rankings_df["chunk_id"].notna().all(), "Есть пустые chunk_id"
assert dense_rankings_df["question_id"].notna().all(), "Есть пустые question_id"

print("Проверка dense_rankings_df пройдена")

Проверка dense_rankings_df пройдена


In [46]:
# Просматриваем ТОП-10 чанков для первых 3-х вопросов.
sample_query_ids = gold_df["question_id"].head(3).tolist()

for qid in sample_query_ids:
    print("=" * 80)
    print("QUESTION_ID:", qid)
    print("QUESTION:", gold_df.loc[gold_df["question_id"] == qid, "question"].iloc[0])
    display(dense_rankings_df[dense_rankings_df["question_id"] == qid].head(10))


QUESTION_ID: 0
QUESTION: 1. Какая выручка компании за 2024 год?


,question_id,method,rank,chunk_id,score
0,0,dense,1,пао омз-p042-c002,0.871737
1,0,dense,2,пао омз-p041-c005,0.870817
2,0,dense,3,глобалтрак менеджмент-p057-c007,0.870680
3,0,dense,4,мечел-p001-c002,0.869142
4,0,dense,5,лента-p064-c002,0.868623
5,0,dense,6,пао афк система-p031-c006,0.868352
6,0,dense,7,пао ятэк-p008-c002,0.867237
7,0,dense,8,мтс-p036-c004,0.867223
8,0,dense,9,русгидро-p060-c004,0.867145
9,0,dense,10,мтс-p043-c004,0.867126


QUESTION_ID: 1
QUESTION: 2. Какую чистую прибыль компания получила за 2024 год?


,question_id,method,rank,chunk_id,score
10,1,dense,1,пао омз-p041-c005,0.879948
11,1,dense,2,пао югк-p015-c002,0.874528
12,1,dense,3,ржд-p013-c003,0.870286
13,1,dense,4,русагро-p036-c004,0.870219
14,1,dense,5,магнит-p066-c007,0.867342
15,1,dense,6,пао омз-p055-c004,0.867068
16,1,dense,7,пао афк система-p031-c006,0.866524
17,1,dense,8,пао омз-p041-c006,0.865307
18,1,dense,9,русагро-p066-c002,0.864268
19,1,dense,10,пао омз-p056-c003,0.863994


QUESTION_ID: 2
QUESTION: 3. Какова общая сумма активов компании на 31 декабря 2024 года?


,question_id,method,rank,chunk_id,score
20,2,dense,1,пао омз-p055-c004,0.892828
21,2,dense,2,лукойл-p024-c002,0.892123
22,2,dense,3,пао омз-p056-c004,0.891909
23,2,dense,4,россети-p061-c003,0.891845
24,2,dense,5,газпром-p097-c002,0.890410
25,2,dense,6,россети-p009-c002,0.890203
26,2,dense,7,сбер-p154-c002,0.889554
27,2,dense,8,мечел-p008-c001,0.888268
28,2,dense,9,лукойл-p038-c002,0.887169
29,2,dense,10,м видео-p076-c003,0.887133


Сохраняем итоговую табличку dense_rankings.csv

In [47]:
# Сохраним
dense_rankings_df.to_csv('dense_rankings.csv', index=False, encoding="utf-8-sig")